# Week 7 — Notebook 4: Stratified K-Fold Cross-Validation

**Difficulty:** ⭐⭐⭐ &nbsp;&nbsp; **Estimated Time:** 2 hours  
**Domain:** Credit Card Fraud Detection  
**Topics:** stratified k-fold · imbalanced datasets · class-proportion preservation · multi-class stratification

---

## Why This Matters

In fraud detection, fraud cases are rare — often 1-5% of all transactions.  
Standard k-fold shuffles data **randomly**, which causes a critical problem:

With 2% fraud across 5000 transactions (100 fraud cases, 4900 legit):  
- 5-fold CV → 5 folds of 1000 transactions each  
- Expected fraud per fold: 20 cases  
- But random shuffling could easily produce: [5, 8, 35, 28, 24]  
- The fold with only 5 fraud cases will yield a very different F1 score than the fold with 35!  

**Stratified k-fold solves this by guaranteeing each fold has the same fraud rate as the full dataset.**  
This dramatically reduces variance in your CV scores for imbalanced data.

## Real-World Analogy: The Proportional Fruit Salad

Imagine you have a large fruit salad with:
- 98 apples (legitimate transactions)
- 2 strawberries (fraud)

You want to divide it equally into **5 bowls** for 5 guests.

**Standard k-fold (random scooping):**  
You scoop randomly. Guest 1 might get both strawberries (and complain about the ratio).  
Guest 2 gets no strawberries at all. The experience is wildly inconsistent.

**Stratified k-fold (careful portioning):**  
You first sort the strawberries. You put 0.4 strawberries worth into each bowl (e.g., alternating which bowl gets a whole strawberry).  
Each guest gets **approximately the same ratio** of each fruit.

```
How stratification works:

Step 1 — Separate classes:      [fraud_1 ... fraud_100]    [legit_1 ... legit_4900]
Step 2 — Divide each class:     fraud split into 5 parts:  [20] [20] [20] [20] [20]
                                 legit split into 5 parts: [980][980][980][980][980]
Step 3 — Combine for each fold: fold_i = fraud_part_i + legit_part_i

Result: each fold has exactly 2% fraud (same as full dataset)
```

## The Algorithm

```
INPUT:  X, y, k
OUTPUT: k fold index sets, each with same class proportions

1. Find unique classes C = {c₁, c₂, ...}
2. For each class cᵢ:
      gather all sample indices where y == cᵢ
      shuffle those indices
      round-robin assign to folds 0, 1, 2, ..., k-1, 0, 1, ...
3. Each fold k gets ≈ (total_class_cᵢ) / k samples from class cᵢ
4. Use these folds exactly like standard k-fold
```

### When Must You Use Stratified K-Fold?

| Situation | Use Stratified? | Why |
|---|---|---|
| 2% fraud, binary classification | **Yes — always** | Random split may have 0% fraud in a fold |
| 50% fraud, balanced dataset | Optional | Less critical, but still good practice |
| Multi-class, rare class (0.5%) | **Yes** | Same reason as binary |
| Regression (continuous y) | No (use other methods) | Stratification requires discrete classes |
| Very large dataset (n > 100k) | Optional | Class imbalance per fold is small by chance |

### Key sklearn classes
```python
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for train_idx, test_idx in skf.split(X, y):    # y is required!
    ...
```

## Setup & Severely Imbalanced Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    KFold, StratifiedKFold, cross_val_score, train_test_split
)
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#e0e0e0',
    'ytick.color':      '#e0e0e0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#333',
    'grid.alpha':       0.5,
    'axes.titlecolor':  '#ffffff',
    'figure.titlesize': 14,
})
PALETTE = ['#00d4ff', '#ff6b6b', '#51cf66', '#ffd43b', '#cc5de8', '#ff922b']

print('Libraries loaded.')

In [ ]:
# ── Create severely imbalanced fraud dataset ──────────────────────────────────
# 5000 transactions, 2% fraud (100 fraud cases) — very realistic for real systems

np.random.seed(42)
N = 5000
FRAUD_RATE = 0.02
n_fraud = int(N * FRAUD_RATE)          # exactly 100 fraud cases
n_legit = N - n_fraud                  # 4900 legitimate transactions

# Feature generation (5 features)
X_legit = np.column_stack([
    np.random.normal(60,  35,  n_legit),   # transaction_amount
    np.random.normal(13,  5,   n_legit),   # hour_of_day (daytime peak)
    np.random.normal(2.5, 1,   n_legit),   # txns_last_hour
    np.random.normal(0.1, 0.08, n_legit),  # foreign_merchant_ratio
    np.random.normal(1.2, 0.4, n_legit),   # velocity_score
])

X_fraud = np.column_stack([
    np.random.normal(250, 100, n_fraud),   # high amounts
    np.random.normal(2,   2,   n_fraud),   # late night / early morning
    np.random.normal(9,   4,   n_fraud),   # many txns in short time
    np.random.normal(0.8, 0.15, n_fraud),  # mostly foreign merchants
    np.random.normal(5,   1.5, n_fraud),   # high velocity
])

X_raw = np.vstack([X_legit, X_fraud])
y     = np.array([0]*n_legit + [1]*n_fraud)

# Shuffle and scale
perm = np.random.permutation(N)
X_raw, y = X_raw[perm], y[perm]
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

feature_names = ['amount', 'hour', 'txns_last_hour', 'foreign_ratio', 'velocity']

print(f'Dataset: {N} transactions')
print(f'  Fraud : {y.sum():>4} ({y.mean()*100:.1f}%)')
print(f'  Legit : {(y==0).sum():>4} ({(y==0).mean()*100:.1f}%)')
print(f'\nImbalance ratio: 1 fraud per {int(1/FRAUD_RATE)} transactions')
print('Expected fraud per fold (k=5): ~20 cases per 1000 transactions')

## Part 1 — Standard K-Fold: Fraud Rate Varies Wildly Per Fold

In [ ]:
# ── Standard KFold: measure fraud rate in each fold ───────────────────────────
# This demonstrates the PROBLEM: random shuffling causes varying fraud rates.

K = 5
kf_standard = KFold(n_splits=K, shuffle=True, random_state=42)

print('=== Standard K-Fold (k=5): Fraud Rate per Fold ===')
print(f'{"Fold":<6} {"N test":>8} {"N fraud":>8} {"Fraud %":>9} {"vs. expected":>14}')
print('-' * 50)

standard_fraud_rates = []
for i, (train_idx, test_idx) in enumerate(kf_standard.split(X, y)):
    y_test_fold = y[test_idx]
    n_test = len(y_test_fold)
    n_fraud_fold = y_test_fold.sum()
    fraud_pct = n_fraud_fold / n_test * 100
    standard_fraud_rates.append(fraud_pct)
    delta = fraud_pct - FRAUD_RATE * 100
    flag  = '<-- HIGH' if delta > 1 else ('<-- LOW' if delta < -1 else '  ok')
    print(f'{i+1:<6} {n_test:>8} {n_fraud_fold:>8} {fraud_pct:>8.2f}%  {delta:>+7.2f}%  {flag}')

expected = FRAUD_RATE * 100
print(f'\nExpected fraud rate in each fold: {expected:.2f}%')
print(f'Actual range: {min(standard_fraud_rates):.2f}% – {max(standard_fraud_rates):.2f}%')
print(f'Std across folds: {np.std(standard_fraud_rates):.3f}%')

## Part 2 — Stratified K-Fold Implemented From Scratch

In [ ]:
# ── Stratified k-fold from scratch ───────────────────────────────────────────
# Algorithm: for each class, shuffle its indices, then round-robin assign to folds.
# Round-robin ensures each fold gets approximately (class_count / k) samples from each class.

def stratified_kfold_split(X, y, k=5, random_state=42):
    """
    Stratified k-fold split.

    Returns a list of k tuples: (train_indices, test_indices)
    Each fold preserves the class proportions of the full dataset.
    """
    rng = np.random.RandomState(random_state)
    classes = np.unique(y)

    # Assign each sample to one of k fold buckets
    fold_buckets = [[] for _ in range(k)]   # fold_buckets[i] = list of sample indices in fold i

    for cls in classes:
        cls_indices = np.where(y == cls)[0]  # all indices belonging to this class
        rng.shuffle(cls_indices)             # shuffle within class (for randomness)
        # Round-robin: assign index j to fold (j % k)
        for j, idx in enumerate(cls_indices):
            fold_buckets[j % k].append(idx)

    # Convert bucket lists to numpy arrays
    fold_indices = [np.array(bucket) for bucket in fold_buckets]

    # Build (train, test) splits
    splits = []
    for i in range(k):
        test_idx  = fold_indices[i]
        train_idx = np.concatenate([fold_indices[j] for j in range(k) if j != i])
        splits.append((train_idx, test_idx))

    return splits


# Run the scratch implementation
stratified_splits = stratified_kfold_split(X, y, k=5, random_state=42)

print('=== Stratified K-Fold (scratch, k=5): Fraud Rate per Fold ===')
print(f'{"Fold":<6} {"N test":>8} {"N fraud":>8} {"Fraud %":>9} {"vs. expected":>14}')
print('-' * 50)

stratified_fraud_rates = []
for i, (train_idx, test_idx) in enumerate(stratified_splits):
    y_test_fold  = y[test_idx]
    n_test       = len(y_test_fold)
    n_fraud_fold = y_test_fold.sum()
    fraud_pct    = n_fraud_fold / n_test * 100
    stratified_fraud_rates.append(fraud_pct)
    delta = fraud_pct - FRAUD_RATE * 100
    print(f'{i+1:<6} {n_test:>8} {n_fraud_fold:>8} {fraud_pct:>8.2f}%  {delta:>+7.2f}%')

print(f'\nExpected fraud rate: {FRAUD_RATE*100:.2f}%')
print(f'Range: {min(stratified_fraud_rates):.2f}% – {max(stratified_fraud_rates):.2f}%')
print(f'Std  : {np.std(stratified_fraud_rates):.4f}%  (should be near 0)')

## Part 3 — Side-by-Side Bar Chart: Fraud Rate Per Fold

In [ ]:
# ── Visualise: fraud rate per fold, standard vs stratified ───────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
fig.suptitle('Fraud Rate per Fold: Standard vs Stratified K-Fold\n'
             '(2% fraud, n=5000, k=5)',
             fontsize=13, fontweight='bold')

fold_labels = [f'Fold {i+1}' for i in range(K)]

for ax, rates, title, color in [
    (axes[0], standard_fraud_rates,   'Standard KFold (random)',      PALETTE[1]),
    (axes[1], stratified_fraud_rates, 'Stratified KFold (our scratch)', PALETTE[0])
]:
    bars = ax.bar(fold_labels, rates, color=color, alpha=0.85, edgecolor='white')
    ax.axhline(FRAUD_RATE*100, color='#ffd43b', lw=2, ls='--',
               label=f'Expected {FRAUD_RATE*100:.1f}%')
    ax.set_ylabel('Fraud Rate in Fold (%)')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, max(max(standard_fraud_rates), 5) * 1.3)

    # Annotate bars with exact rates
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{rate:.2f}%', ha='center', va='bottom', fontsize=9, color='white')

# Add std annotation
axes[0].set_title(
    f'Standard KFold\nstd={np.std(standard_fraud_rates):.3f}%  (high!)', fontsize=11)
axes[1].set_title(
    f'Stratified KFold\nstd={np.std(stratified_fraud_rates):.4f}%  (near 0)', fontsize=11)

plt.tight_layout()
plt.savefig('stratified_fraud_rate_per_fold.png', dpi=120, bbox_inches='tight')
plt.show()

## Part 4 — Metric Variance: F1 Scores Fold-by-Fold

In [ ]:
# ── Compare F1 per fold: standard k-fold vs stratified k-fold ────────────────
# F1 is sensitive to class balance: folds with few fraud cases → wildly different F1.

def run_cv_get_fold_scores(splits, X, y, metric='f1'):
    """Run CV using pre-computed splits; return per-fold scores."""
    scores = []
    for train_idx, test_idx in splits:
        model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
        model.fit(X[train_idx], y[train_idx])
        y_pred = model.predict(X[test_idx])
        y_prob = model.predict_proba(X[test_idx])[:, 1]
        if metric == 'f1':
            scores.append(f1_score(y[test_idx], y_pred, zero_division=0))
        elif metric == 'roc_auc':
            try:
                scores.append(roc_auc_score(y[test_idx], y_prob))
            except ValueError:
                scores.append(np.nan)           # only one class in fold
    return np.array(scores)


# Build standard splits manually to match shape expected by run_cv_get_fold_scores
kf_standard_splits = list(kf_standard.split(X, y))

f1_standard    = run_cv_get_fold_scores(kf_standard_splits,  X, y, metric='f1')
f1_stratified  = run_cv_get_fold_scores(stratified_splits,   X, y, metric='f1')
auc_standard   = run_cv_get_fold_scores(kf_standard_splits,  X, y, metric='roc_auc')
auc_stratified = run_cv_get_fold_scores(stratified_splits,   X, y, metric='roc_auc')

print(f'{"":28} {"Standard KFold":>16} {"Stratified KFold":>18}')
print('-' * 65)
for i in range(K):
    print(f'  Fold {i+1} — F1 / AUC :     '
          f'{f1_standard[i]:.4f} / {auc_standard[i]:.4f}    '
          f'{f1_stratified[i]:.4f} / {auc_stratified[i]:.4f}')
print('-' * 65)
print(f'  Mean                  '
      f'{np.nanmean(f1_standard):.4f} / {np.nanmean(auc_standard):.4f}    '
      f'{np.mean(f1_stratified):.4f} / {np.mean(auc_stratified):.4f}')
print(f'  Std                   '
      f'{np.nanstd(f1_standard):.4f} / {np.nanstd(auc_standard):.4f}    '
      f'{np.std(f1_stratified):.4f} / {np.std(auc_stratified):.4f}')

In [ ]:
# ── Plot F1 per fold ──────────────────────────────────────────────────────────

x_pos    = np.arange(K)
width    = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('F1 and AUC Per Fold: Standard vs Stratified K-Fold\n'
             '(2% fraud, Logistic Regression with class_weight=balanced)',
             fontsize=13, fontweight='bold')

for ax, std_vals, strat_vals, metric_name in [
    (axes[0], f1_standard,  f1_stratified,  'F1 Score'),
    (axes[1], auc_standard, auc_stratified, 'ROC-AUC'),
]:
    bars1 = ax.bar(x_pos - width/2, std_vals,   width,
                   label='Standard KFold', color=PALETTE[1], alpha=0.85, edgecolor='white')
    bars2 = ax.bar(x_pos + width/2, strat_vals, width,
                   label='Stratified KFold', color=PALETTE[0], alpha=0.85, edgecolor='white')

    ax.axhline(np.nanmean(std_vals),   color=PALETTE[1], lw=1.5, ls='--', alpha=0.6)
    ax.axhline(np.mean(strat_vals),    color=PALETTE[0], lw=1.5, ls='--', alpha=0.6)

    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'Fold {i+1}' for i in range(K)])
    ax.set_ylabel(metric_name)
    ax.set_title(f'{metric_name} per Fold  '
                 f'(std: standard={np.nanstd(std_vals):.3f}, '
                 f'stratified={np.std(strat_vals):.3f})', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('stratified_f1_auc_per_fold.png', dpi=120, bbox_inches='tight')
plt.show()
print('Key: Stratified KFold produces far more consistent F1/AUC across folds.')

## Part 5 — Extreme Imbalance: 0.5% Fraud (Standard KFold Fails)

In [ ]:
# ── Extreme imbalance: 0.5% fraud — some standard folds have ZERO fraud ───────
# This is where standard k-fold completely breaks down:
#   - A fold with 0 fraud cases cannot compute F1 (no positive class)
#   - AUC is undefined when only one class is present

N_EXT   = 2000
N_FRAUD_EXT = 10            # 0.5% of 2000 = 10 fraud cases
n_legit_ext = N_EXT - N_FRAUD_EXT

np.random.seed(7)
X_ext = np.vstack([
    np.random.randn(n_legit_ext, 4),
    np.random.randn(N_FRAUD_EXT, 4) + 2
])
y_ext = np.array([0]*n_legit_ext + [1]*N_FRAUD_EXT)
perm_ext = np.random.permutation(N_EXT)
X_ext, y_ext = X_ext[perm_ext], y_ext[perm_ext]

K_EXT = 5
kf_ext  = KFold(n_splits=K_EXT, shuffle=True, random_state=42)
skf_ext = StratifiedKFold(n_splits=K_EXT, shuffle=True, random_state=42)

print(f'Dataset: {N_EXT} samples, {N_FRAUD_EXT} fraud ({N_FRAUD_EXT/N_EXT*100:.1f}%)')
print(f'Expected fraud per fold (k={K_EXT}): {N_FRAUD_EXT/K_EXT:.1f} cases')
print()
print(f'{"":6} {"Standard KFold":>18} {"Stratified KFold":>20}')
print(f'{"Fold":<6} {"N fraud":>10} {"Fraud%":>8} {"N fraud":>12} {"Fraud%":>8}')
print('-' * 58)

zero_fraud_folds = 0
for i, ((tr1, te1), (tr2, te2)) in enumerate(
    zip(kf_ext.split(X_ext, y_ext), skf_ext.split(X_ext, y_ext))
):
    nf_std   = y_ext[te1].sum()
    nf_strat = y_ext[te2].sum()
    pct_std   = nf_std  / len(te1)  * 100
    pct_strat = nf_strat/ len(te2) * 100
    warn = '  <-- ZERO FRAUD!' if nf_std == 0 else ''
    if nf_std == 0:
        zero_fraud_folds += 1
    print(f'{i+1:<6} {nf_std:>10} {pct_std:>7.2f}%  {nf_strat:>12} {pct_strat:>7.2f}%  {warn}')

print()
print(f'Standard KFold folds with 0 fraud: {zero_fraud_folds}/{K_EXT}')
print('Stratified KFold folds with 0 fraud: 0 (guaranteed)')
print()
print('Impact: a fold with 0 fraud cannot compute F1 or AUC!')
print('The model appears to have undefined performance for that fold.')

## Part 6 — Multi-Class Stratification: Email Category Dataset

In [ ]:
# ── 4-class email dataset; verify stratification preserves each class per fold ──
# Classes and sizes (deliberately unequal to stress-test stratification):
#   Class 0 — Spam:        50%
#   Class 1 — Promotions:  25%
#   Class 2 — Social:      15%
#   Class 3 — Phishing:    10%  (rare class — most vulnerable without stratification)

np.random.seed(0)
N_MC = 1000
class_props = [0.50, 0.25, 0.15, 0.10]
class_names = ['Spam', 'Promotions', 'Social', 'Phishing']

y_mc = np.concatenate([
    np.full(int(N_MC * p), c)
    for c, p in enumerate(class_props)
]).astype(int)
y_mc = y_mc[:N_MC]   # trim if rounding caused extra
X_mc = np.random.randn(len(y_mc), 10)

perm_mc = np.random.permutation(len(y_mc))
X_mc, y_mc = X_mc[perm_mc], y_mc[perm_mc]

print('Multi-class email dataset:')
for c, name in enumerate(class_names):
    count = (y_mc == c).sum()
    print(f'  Class {c} ({name:>12}): {count:>4} samples ({count/len(y_mc)*100:.1f}%)')

print(f'Total: {len(y_mc)} samples')

In [ ]:
# ── Compare per-class proportions in each fold ────────────────────────────────

K_MC = 5
kf_mc  = KFold(n_splits=K_MC, shuffle=True, random_state=42)
skf_mc = StratifiedKFold(n_splits=K_MC, shuffle=True, random_state=42)

def fold_class_proportions(splits, y, n_classes):
    """Return matrix (k x n_classes) of class proportions in each test fold."""
    props = []
    for _, test_idx in splits:
        fold_counts = np.array([(y[test_idx] == c).sum() for c in range(n_classes)])
        props.append(fold_counts / len(test_idx))
    return np.array(props)

props_standard   = fold_class_proportions(list(kf_mc.split(X_mc, y_mc)),  y_mc, 4)
props_stratified = fold_class_proportions(list(skf_mc.split(X_mc, y_mc)), y_mc, 4)
target_props     = np.array(class_props)

print('Class proportions per fold (Standard KFold):')
print(f'{"Fold":<6}' + ''.join(f'{name:>14}' for name in class_names))
for i, row in enumerate(props_standard):
    print(f'{i+1:<6}' + ''.join(f'{v*100:>13.1f}%' for v in row))
print(f'{"Target":<6}' + ''.join(f'{p*100:>13.1f}%' for p in target_props))

print()
print('Class proportions per fold (Stratified KFold):')
print(f'{"Fold":<6}' + ''.join(f'{name:>14}' for name in class_names))
for i, row in enumerate(props_stratified):
    print(f'{i+1:<6}' + ''.join(f'{v*100:>13.1f}%' for v in row))
print(f'{"Target":<6}' + ''.join(f'{p*100:>13.1f}%' for p in target_props))

In [ ]:
# ── Stacked bar chart: class proportions per fold ─────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle('Multi-Class Stratification: Class Proportions per Fold\n'
             '4-class Email Dataset (Spam 50%, Promo 25%, Social 15%, Phishing 10%)',
             fontsize=12, fontweight='bold')

colors_mc = [PALETTE[0], PALETTE[2], PALETTE[3], PALETTE[1]]

for ax, props, title in [
    (axes[0], props_standard,   'Standard KFold (random)'),
    (axes[1], props_stratified, 'Stratified KFold'),
]:
    bottoms = np.zeros(K_MC)
    fold_labels_mc = [f'Fold {i+1}' for i in range(K_MC)]

    for c, (cname, color) in enumerate(zip(class_names, colors_mc)):
        vals = props[:, c] * 100
        bars = ax.bar(fold_labels_mc, vals, bottom=bottoms,
                      label=cname, color=color, alpha=0.85, edgecolor='white')
        # Annotate each segment
        for bar, v, bot in zip(bars, vals, bottoms):
            if v > 3:
                ax.text(bar.get_x() + bar.get_width()/2,
                        bot + v/2, f'{v:.0f}%',
                        ha='center', va='center', fontsize=8, color='white', fontweight='bold')
        bottoms = bottoms + vals

    # Add target dashed lines for key boundaries
    cumsum = np.cumsum([p*100 for p in class_props])
    for boundary, ls in zip(cumsum[:-1], ['--', '-.', ':']):
        ax.axhline(boundary, color='#ffd43b', ls=ls, lw=1, alpha=0.7)

    ax.set_ylabel('Class Proportion (%)')
    ax.set_title(title, fontsize=11)
    ax.legend(loc='upper right', fontsize=8, ncol=2)
    ax.set_ylim(0, 115)
    ax.grid(False)

plt.tight_layout()
plt.savefig('stratified_multiclass.png', dpi=120, bbox_inches='tight')
plt.show()

## Part 7 — sklearn Comparison: StratifiedKFold vs Manual Implementation

In [ ]:
# ── Verify sklearn StratifiedKFold produces same fraud rates as our scratch ───
# (The exact splits may differ because sklearn uses a different tie-breaking
#  strategy in the round-robin, but the fraud rates should be essentially identical.)

skf_sklearn = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sklearn_splits = list(skf_sklearn.split(X, y))

print('=== Fraud Rate Per Fold: sklearn StratifiedKFold vs Scratch ===')
print(f'{"Fold":<6} {"sklearn":>10} {"Scratch":>10} {"Difference":>12}')
print('-' * 42)

sklearn_fraud_rates = []
for i, (_, te) in enumerate(sklearn_splits):
    rate = y[te].mean() * 100
    sklearn_fraud_rates.append(rate)

for i, (sk, sc) in enumerate(zip(sklearn_fraud_rates, stratified_fraud_rates)):
    diff = abs(sk - sc)
    match = 'exact match' if diff < 0.01 else f'diff={diff:.3f}%'
    print(f'{i+1:<6} {sk:>9.3f}%  {sc:>9.3f}%  {match:>12}')

print()
print(f'sklearn std of fraud rates: {np.std(sklearn_fraud_rates):.4f}%')
print(f'Scratch std of fraud rates: {np.std(stratified_fraud_rates):.4f}%')
print()

# Run full CV with sklearn StratifiedKFold
scores_skf = cross_val_score(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    X, y, cv=skf_sklearn, scoring='f1'
)
scores_manual = run_cv_get_fold_scores(sklearn_splits, X, y, metric='f1')

print('F1 from cross_val_score (sklearn StratifiedKFold):')
print(f'  {scores_skf.round(4)}  mean={scores_skf.mean():.4f} ± {scores_skf.std():.4f}')
print()
print('F1 from our scratch run_cv_get_fold_scores (same sklearn splits):')
print(f'  {scores_manual.round(4)}  mean={scores_manual.mean():.4f} ± {scores_manual.std():.4f}')

## Part 8 — Stratified Train/Test Split & Summary Dashboard

In [ ]:
# ── Stratified train_test_split ───────────────────────────────────────────────
# Always use stratify=y when splitting an imbalanced dataset.
# This applies stratification to a two-way split (not k-fold).

# Without stratification
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
fraud_train_unstrat = y_tr.mean() * 100
fraud_test_unstrat  = y_te.mean() * 100

# With stratification
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
fraud_train_strat = y_tr_s.mean() * 100
fraud_test_strat  = y_te_s.mean() * 100

print(f'Dataset overall fraud rate: {y.mean()*100:.2f}%')
print()
print('Without stratify=y:')
print(f'  Train fraud rate: {fraud_train_unstrat:.2f}%')
print(f'  Test  fraud rate: {fraud_test_unstrat:.2f}%')
print(f'  Gap: {abs(fraud_train_unstrat - fraud_test_unstrat):.3f}%')
print()
print('With stratify=y:')
print(f'  Train fraud rate: {fraud_train_strat:.2f}%')
print(f'  Test  fraud rate: {fraud_test_strat:.2f}%')
print(f'  Gap: {abs(fraud_train_strat - fraud_test_strat):.3f}%  (near zero)')
print()
print('Rule of thumb: ALWAYS pass stratify=y for any imbalanced classification split.')

In [ ]:
# ── Summary dashboard ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Stratified K-Fold — Summary Dashboard\nCredit Card Fraud Detection (2%)',
             fontsize=14, fontweight='bold')

# ① Fraud rate per fold comparison
ax0 = axes[0, 0]
x_f = np.arange(K)
w_f = 0.35
ax0.bar(x_f - w_f/2, standard_fraud_rates,   w_f, label='Standard KFold',   color=PALETTE[1], alpha=0.85, edgecolor='white')
ax0.bar(x_f + w_f/2, stratified_fraud_rates, w_f, label='Stratified KFold', color=PALETTE[0], alpha=0.85, edgecolor='white')
ax0.axhline(FRAUD_RATE*100, color='#ffd43b', lw=2, ls='--', label='Target 2.0%')
ax0.set_xticks(x_f)
ax0.set_xticklabels([f'F{i+1}' for i in range(K)])
ax0.set_ylabel('Fraud Rate (%)')
ax0.set_title('① Fraud Rate per Fold', fontsize=11)
ax0.legend(fontsize=8)
ax0.grid(True, alpha=0.3, axis='y')

# ② F1 per fold comparison
ax1 = axes[0, 1]
ax1.bar(x_f - w_f/2, f1_standard,   w_f, label='Standard KFold',   color=PALETTE[1], alpha=0.85, edgecolor='white')
ax1.bar(x_f + w_f/2, f1_stratified, w_f, label='Stratified KFold', color=PALETTE[0], alpha=0.85, edgecolor='white')
ax1.axhline(np.nanmean(f1_standard),  color=PALETTE[1], lw=1.5, ls='--', alpha=0.7)
ax1.axhline(np.mean(f1_stratified),   color=PALETTE[0], lw=1.5, ls='--', alpha=0.7)
ax1.set_xticks(x_f)
ax1.set_xticklabels([f'F{i+1}' for i in range(K)])
ax1.set_ylabel('F1 Score')
ax1.set_title(f'② F1 per Fold\n'
              f'(std: std={np.nanstd(f1_standard):.3f} vs strat={np.std(f1_stratified):.3f})',
              fontsize=10)
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_ylim(0, 1.1)

# ③ How stratification works (text diagram)
ax2 = axes[1, 0]
ax2.axis('off')
ax2.set_title('③ How Stratification Works', fontsize=11)
diagram_text = (
    'Full dataset: 98 legit + 2 fraud\n\n'
    'Standard (random):          Stratified (proportional):\n'
    'Fold1: 20 legit + 0 fraud   Fold1: 19 legit + ~0.4 fraud\n'
    'Fold2: 19 legit + 2 fraud   Fold2: 20 legit + ~0.4 fraud\n'
    'Fold3: 21 legit + 0 fraud   Fold3: 19 legit + ~0.4 fraud\n'
    '  ...                         ...\n\n'
    'Algorithm:\n'
    '1. Separate classes\n'
    '2. Shuffle within each class\n'
    '3. Round-robin assign to folds\n'
    '4. Each fold gets ≈ class_n / k samples'
)
ax2.text(0.05, 0.9, diagram_text, transform=ax2.transAxes,
         va='top', ha='left', fontsize=9, family='monospace',
         color='#e0e0e0',
         bbox=dict(boxstyle='round', facecolor='#1a1a2e', edgecolor='#444', alpha=0.8))

# ④ Multi-class stacked bar (summary)
ax3 = axes[1, 1]
bottoms4 = np.zeros(K_MC)
fold_labels4 = [f'F{i+1}' for i in range(K_MC)]
for c, (cname, color) in enumerate(zip(class_names, colors_mc)):
    vals4 = props_stratified[:, c] * 100
    ax3.bar(fold_labels4, vals4, bottom=bottoms4, label=cname,
            color=color, alpha=0.85, edgecolor='white')
    for fi, (v, b) in enumerate(zip(vals4, bottoms4)):
        if v > 3:
            ax3.text(fi, b + v/2, f'{v:.0f}%', ha='center', va='center',
                     fontsize=8, color='white', fontweight='bold')
    bottoms4 = bottoms4 + vals4
ax3.set_title('④ Multi-Class Stratification (4 email classes)', fontsize=10)
ax3.set_ylabel('Class Proportion (%)')
ax3.legend(fontsize=8, ncol=2)
ax3.grid(False)

plt.tight_layout()
plt.savefig('stratified_summary_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

## Key Takeaways

| Concept | Plain English |
|---|---|
| **The problem** | Standard k-fold may put most fraud in one fold by chance → wildly varying F1/AUC |
| **Stratification** | Ensures each fold has exactly the same fraud rate as the full dataset |
| **How it works** | Round-robin assignment within each class; guarantees proportional representation |
| **When to always use it** | Imbalanced binary/multi-class, rare classes, when using F1/AUC (not accuracy) |
| **Extreme case** | 0.5% fraud + standard k-fold → folds with zero fraud → AUC undefined! |
| **Multi-class** | Stratification preserves all class proportions simultaneously |
| **train_test_split** | Use `stratify=y` — same principle, single split |
| **sklearn** | `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` |

### Practical Decision Guide
```
Is this classification?
  No (regression)    → Use standard KFold or KFold with value binning
  Yes
    Is data balanced (each class > 10%)?
      Yes → standard KFold is fine, but StratifiedKFold never hurts
      No (any class < 10%)
        → ALWAYS use StratifiedKFold
        → ALWAYS use stratify=y in train_test_split
        → ALWAYS use F1/AUC, NOT accuracy
```

## Self-Check Questions

---

**Question 1**  
You have 1000 samples and 10 fraud cases. With standard 5-fold CV, what is the expected number of fraud cases in each fold? What is the minimum that could realistically occur by chance?

<details>
<summary>Click to reveal answer</summary>

**Expected:** Each fold has 200 samples. With 10 fraud cases uniformly distributed, you would expect 10 / 5 = **2 fraud cases per fold**.

**Minimum by chance:** Since 2 per fold is the average, random allocation can easily put 0 fraud cases in a fold. Consider: 10 fraud cases distributed randomly across 5 folds of 200 samples. The probability of a specific fold having 0 fraud follows a hypergeometric distribution:

P(0 fraud in fold) = C(990, 200) / C(1000, 200) ≈ 0.12 (about 12% chance)

This means roughly 1 in 8 runs of 5-fold CV will have at least one fold with zero fraud cases!  
**Stratified k-fold guarantees 2 fraud per fold (±1 for rounding).**

</details>

---

**Question 2**  
Your 5-fold CV reports F1 scores: [0.91, 0.45, 0.88, 0.92, 0.87]. What does the outlier fold (0.45) likely indicate?

<details>
<summary>Click to reveal answer</summary>

The outlier fold (F1 = 0.45) is almost certainly caused by **class imbalance within that fold** — most likely a fold that received very few (or zero) fraud cases in its test set.

Possible causes:
1. **Standard k-fold was used instead of stratified k-fold** — the random shuffle put almost all fraud cases into the other four folds, leaving fold 2 with almost no fraud. A model cannot be evaluated for fraud detection if there's no fraud to detect.
2. **Very small minority class overall** — even with stratification, if you have only 5 fraud cases total and k=5, each fold gets ~1 fraud case. A single misclassification tanks F1 dramatically.
3. **Data leakage or a corrupted fold** (less common).

**Fix:** Switch to `StratifiedKFold`. The F1 scores should become much more consistent (e.g., [0.89, 0.90, 0.87, 0.91, 0.88]).  

**Important:** Do NOT simply average [0.91, 0.45, 0.88, 0.92, 0.87] and report 0.81 as your model's F1. The 0.45 is a measurement artifact, not a real data point.

</details>

---

**Question 3**  
For a regression problem (predicting continuous values like transaction dollar amount), should you use stratified k-fold? If not, what could you do to get the benefits of stratification?

<details>
<summary>Click to reveal answer</summary>

**No** — `StratifiedKFold` requires discrete class labels and cannot be directly applied to continuous targets. If you pass continuous y to `StratifiedKFold`, sklearn will raise a `ValueError`.

**Alternatives to stratify a regression problem:**

1. **Bin the target into quantiles, then stratify by bin:**
   ```python
   y_binned = pd.qcut(y, q=10, labels=False)   # 10 equal-frequency bins
   skf = StratifiedKFold(n_splits=5, shuffle=True)
   for train_idx, test_idx in skf.split(X, y_binned):  # use binned y for splitting
       model.fit(X[train_idx], y[train_idx])            # but train/eval on continuous y
   ```
   This ensures each fold has similar distributions of small, medium, and large values.

2. **Use standard KFold** — for regression, random splits are usually fine because the target is continuous and unlikely to cluster in a pathological way (unlike rare fraud labels).

3. **For time-series regression:** use `TimeSeriesSplit` instead, which respects temporal ordering (no future data leaking into the past).

The fraud detection analogy: if you were predicting *fraud amount* (continuous) rather than *is fraud* (binary), you'd bin fraud amounts (e.g., small/medium/large) and stratify by those bins.

</details>